<div style="text-align: center; line-height: 0; padding-top: 9px;">
<img src="https://learningjournal.github.io/pub-resources/logos/scholarnest_academy.jpg" alt="ScholarNest Academy" style="width: 1400px">
</div>

####1. Read data from the sales_sample.csv file and analyse to identify problems

1.1 Define schema

In [0]:
file_schema = """
id int,
name string,
dop string,
phone long,
amount string,
discount string
"""

1.2 Read data

In [0]:
sales_raw_df = (
    spark.read.format("csv")
        .option("header", "true")
        .schema(file_schema)
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/sales_sample.csv")
)

sales_raw_df.display()

id,name,dop,phone,amount,discount
100,Prashant,2020-06-15,9238614990,12000,18.5
101,David,2018-08-7,8908617610,15000,nil
102,Simran,14-05-2019,null,3000000000,21


1.3 Describe the data

In [0]:
sales_raw_df.describe().display()

summary,id,name,dop,phone,amount,discount
count,3,3,3,2,3,3
mean,101.0,null,null,9.0736163E9,1.000009E9,19.75
stddev,1.0,null,null,2.3334338517179397E8,1.7320430133408928E9,1.7677669529663689
min,100,David,14-05-2019,8908617610,12000,18.5
max,102,Simran,2020-06-15,9238614990,3000000000,nil


1.4 List down the problems you want to fix
1. Convert id from integer to string and rename it as transaction_id.
2. Rename the name column to customer_name.
3. Convert the dop to date format and rename the column to date_of_purchase.
4. Rename the phone column to customer_phone
5. Convert the amount to a long value and filter out nulls and outlier values. 
6. Rename the column to purchase_amount
7. Convert discount to double, converting nil and null values to zero. rename the column to applied_discount

####2. Prepare and clean the Dataframe using appropriate transformations

2.1 Transform

In [0]:
sales_df = sales_raw_df.selectExpr(
    "cast(id as string) as transaction_id",
    "name as customer_name",
    "nvl(try_cast(dop as date), to_date(dop, 'dd-MM-yyyy')) as date_of_purchase",
    "cast(phone as string) as customer_phone",
    "cast(amount as long) as purchase_amount",
    "nvl(try_cast(discount as double), 0) as applied_discount" # try to convert it to double but if its null then give me 0 but if fails conver it to null
).filter("purchase_amount is not null and purchase_amount < 200000") # "where" and "filter" are teh same in spark

sales_df.display()

transaction_id,customer_name,date_of_purchase,customer_phone,purchase_amount,applied_discount
100,Prashant,2020-06-15,9238614990,12000,18.5
101,David,2018-08-07,8908617610,15000,0.0


2.2 Verify statistics

In [0]:
sales_df.describe("purchase_amount", "applied_discount").display()

summary,purchase_amount,applied_discount
count,2,2
mean,13500.0,9.25
stddev,2121.3203435596424,13.08147545195113
min,12000,0.0
max,15000,18.5


&copy; 2021-2026 <a href="https://www.scholarnest.com/">ScholarNest</a>. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the <a href="https://www.apache.org/">Apache Software Foundation.</a><br/>
Databricks, Databricks Cloud and the Databricks logo are trademarks of the <a href="https://www.databricks.com/">Databricks Inc.</a><br/>
<a href="https://www.scholarnest.com/pages/privacy">Privacy Policy</a> | <a href="https://www.scholarnest.com/pages/terms">Terms of Use</a> | <a href="https://www.scholarnest.com/pages/contact">Contact Us</a>

